WEEK 7 Assignment

Step 1: Import Required Libraries

In [1]:
!pip install -q delta-spark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.9 MB/s eta 0:00:00


In [3]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Week7_DeltaLake") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.3


Step 2: Load the Dataset

In [12]:
df = spark.read.csv(
    "/content/superstore.csv",
    header=True,
    inferSchema=True,
    encoding="ISO-8859-1"
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [13]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



Step 4. Save Dataset as Delta Table


In [16]:
delta_path = "/content/delta/superstore"

# Rename columns to remove invalid characters (spaces) for Delta Lake
import re
for col_name in df.columns:
    if ' ' in col_name:
        new_col_name = re.sub(r'[^a-zA-Z0-9_]', '_', col_name)
        df = df.withColumnRenamed(col_name, new_col_name)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

In [17]:
delta_df = spark.read.format("delta").load(delta_path)

delta_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

Step 5. Basic Cleaning

In [18]:
clean_df = delta_df.dropna()

For Removing Duplicate Records

In [19]:
clean_df = clean_df.dropDuplicates()

In [20]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

Step 6. Creating Incremental Dataset

In [21]:
incremental_df = clean_df.limit(5)

In [22]:
from pyspark.sql.functions import lit

incremental_df = incremental_df.withColumn(
    "Sales",
    lit(9999.99)
)

In [23]:
new_record = spark.createDataFrame([
(
10001,
"CA-2027-111111",
"1/1/2027",
"1/5/2027",
"Second Class",
"CG-99999",
"Anushka",
"Consumer",
"United States",
"Nashik",
"Maharashtra",
422001,
"West",
"FUR-NEW",
"Furniture",
"Chairs",
"Office Chair",
5500.0,
2,
0.0,
900.0
)
], clean_df.columns)

incremental_df = incremental_df.union(new_record)

In [24]:
incremental_df.show()

+------+--------------+----------+----------+--------------+-----------+--------------+---------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID| Customer_Name|  Segment|      Country|         City|       State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+--------------+---------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|    37|CA-2016-117590| 12/8/2016|12/10/2016|   First Class|   GH-14485|     Gene Hale|Corporate|United States|   Richardson|       Texas|      75080|Central|FUR-FU-10003664|      Furniture| Furnishings|Electrix Architec

Step 7. MERGE Operation

In [25]:
from delta.tables import DeltaTable

In [26]:
delta_table = DeltaTable.forPath(spark, delta_path)

In [29]:
(
delta_table.alias("target")
.merge(
incremental_df.alias("source"),
"target.Order_ID = source.Order_ID"
)
.whenMatchedUpdate(set={
"Sales":"source.Sales",
"Profit":"source.Profit"
})
.whenNotMatchedInsertAll()
.execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Step 8. Result

In [30]:
final_df = spark.read.format("delta").load(delta_path)

In [31]:
print("Total Rows :", final_df.count())

Total Rows : 9995


In [32]:
duplicates = final_df.count() - final_df.dropDuplicates().count()

print("Duplicate Records :", duplicates)

Duplicate Records : 0


In [33]:
from pyspark.sql.functions import col, count, when

final_df.select([
count(when(col(c).isNull(), c)).alias(c)
for c in final_df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row_ID|Order_ID|Order_Date|Ship_Date|Ship_Mode|Customer_ID|Customer_Name|Segment|Country|City|State|Postal_Code|Region|Product_ID|Category|Sub-Category|Product_Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



Step 9.Final Dataset

In [34]:
final_df.show(20, truncate=False)

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+--------------+------------+-----------+-------+---------------+---------------+------------+-------------------------------------------------------------------------------------+------------+--------+--------+--------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name   |Segment    |Country      |City          |State       |Postal_Code|Region |Product_ID     |Category       |Sub-Category|Product_Name                                                                         |Sales       |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+--------------+------------+-----------+-------+---------------+---------------+------------+-------------------------------------------------------------------------------------+------------+--------+--------+-----

In [35]:
print("Original Rows :", df.count())
print("Rows After Cleaning :", clean_df.count())
print("Rows After Merge :", final_df.count())

Original Rows : 9994
Rows After Cleaning : 9995
Rows After Merge : 9995


Loaded the Superstore dataset into a Delta table.
Cleaned the data by removing null values and duplicate records.
Created an incremental dataset containing updated and new records.
Used the Delta Lake MERGE operation to update existing records and insert new records.
Validated the final dataset by checking the total row count, duplicate records, and null values.
Displayed the final Delta table along with a summary of the processing steps.

Outcome:
The assignment demonstrated how Delta Lake supports efficient incremental data processing by combining update and insert operations into a single MERGE statement while maintaining data consistency.